# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Clone repo cá nhân (fork) vào Colab và chuyển vào thư mục dự án
import os
REPO = 'https://github.com/taquangdung123/K4-Track-4-Day10-VisionLanguageModel.git'
if not os.path.exists('/content/K4-Track-4-Day10-VisionLanguageModel'):
    !git clone {REPO} /content/K4-Track-4-Day10-VisionLanguageModel
%cd /content/K4-Track-4-Day10-VisionLanguageModel

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# Mount Google Drive để sao lưu adapter LoRA / checkpoint sau mỗi lần chạy
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lab_day10'
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# Kiểm tra công cụ của đề: kỳ vọng `68 passed, 1 skipped`
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Bước 1 · Kiểm tra dữ liệu: thống kê từng split, rò rỉ session, vẽ 6 ảnh có box
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
from scripts.labcommon import CLASSES, read_jsonl

DATA = Path('data')
splits = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in ('train', 'val', 'test')}
meta = json.loads((DATA / 'splits' / 'meta.json').read_text(encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])
assert meta['dataset_version'].startswith('d10-')

for s, rows in splits.items():
    boxes = Counter(o['label'] for r in rows for o in r['objects'])
    n_empty = sum(not r['objects'] for r in rows)
    print(f"{s:5s} {len(rows):5d} ảnh | {sum(boxes.values()):4d} box "
          f"{dict((c, boxes[c]) for c in CLASSES)} | ảnh không box: {n_empty} | "
          f"{len({r['session'] for r in rows})} session")

# Không session nào được nằm ở hai split
sess = {s: {r['session'] for r in rows} for s, rows in splits.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = sess[a] & sess[b]
    assert not overlap, f'Rò rỉ session {a}/{b}: {overlap}'
print('OK: không session nào xuất hiện ở hai split')

# Vẽ 6 ảnh train có box (nhãn YOLO chuẩn hoá [0,1] → pixel)
COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
def draw(row):
    img = Image.open(DATA / row['image_path']).convert('RGB')
    W, H = img.size
    d = ImageDraw.Draw(img)
    for o in row['objects']:
        xc, yc, w, h = o['yolo']
        box = ((xc - w/2) * W, (yc - h/2) * H, (xc + w/2) * W, (yc + h/2) * H)
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 2, box[1] + 2), o['label'], fill=COLORS[o['label']])
    d.text((4, 4), row['session'], fill='white')
    return img

random.seed(0)
samples = random.sample([r for r in splits['train'] if r['objects']], 6)
tiles = [draw(r) for r in samples]
W, H = tiles[0].size
grid = Image.new('RGB', (W * 3, H * 2))
for i, t in enumerate(tiles):
    grid.paste(t, ((i % 3) * W, (i // 3) * H))
Path('report/figures').mkdir(parents=True, exist_ok=True)
grid.save('report/figures/step1_train_samples.jpg', quality=85)
grid

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9